# MFC Decomposition over `TARGET_BOX`

This notebook estimates whether ENSO-related moisture flux convergence (MFC) changed between:
- `P1 = 1981-2006`
- `P2 = 2007-2025`

Target box:
- lon `95E-125E`
- lat `6S-2N`

Inputs:
- ERA5 pressure-level `q`, `u`, `v`
- DJF seasonal means, where `DJF 1981 = Dec 1980 + Jan/Feb 1981`
- Nino3.4 DJF index standardized over the full DJF analysis period (`1981-2025`)

The decomposition follows the period-specific climatology in the equations:
- `qbar`, `ubar`, `vbar` are computed separately for each period before anomalies are formed.

Outputs:
- `moisture_decomp_area_mean_TARGETBOX.csv`
- `moisture_decomp_area_mean_TARGETBOX.png`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

REPO_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing').resolve()
DATA_DIR = REPO_ROOT / 'external' / 'ClimateData'
ERA5_DIR = DATA_DIR / 'era5-monthly'
INDEX_DIR = DATA_DIR / 'index-monthly'
NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'dekomposisi_mfc'
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

Q_PATH = ERA5_DIR / 'specific_humidity_1980-2025_1000hpa-100hpa.nc'
UV_PATH = ERA5_DIR / 'uv_wind_1980-2025_1000hpa-100hpa.nc'
NINO34_PATH = INDEX_DIR / 'nino34.anom.csv'
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

OUT_CSV = NOTEBOOK_DIR / 'moisture_decomp_area_mean_TARGETBOX.csv'
OUT_PNG = NOTEBOOK_DIR / 'moisture_decomp_area_mean_TARGETBOX.png'

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

START = '1980-12-01'
END = '2025-02-28'
FULL_YEARS = np.arange(1981, 2026)
P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
DJF_MONTHS = (12, 1, 2)

G = 9.80665
R_EARTH = 6371000.0

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## Helpers and physical meaning

- `total`: regression of the full moisture flux `q*u` or `q*v`
- `dynamic`: circulation change acting on mean moisture
- `thermodynamic`: moisture change acting on mean circulation
- `nonlinear`: interaction between moisture and wind anomalies
- Positive MFC means moisture convergence


In [ ]:
def djf_seasonal_mean(da):
    # Compute DJF means and assign December to the following DJF year.
    da = da.sel(time=slice(START, END))
    month_mask = da.time.dt.month.isin(DJF_MONTHS)
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.sel(time=month_mask).assign_coords(
        djf_year=('time', djf_year.sel(time=month_mask).data)
    )
    da = da.sel(time=da.djf_year.isin(FULL_YEARS))
    return da.groupby('djf_year').mean('time')


def regress_1d(y, x, dim='djf_year'):
    # Ordinary least-squares slope of y onto x along the DJF-year dimension.
    y, x = xr.align(y, x, join='inner')
    x_anom = x - x.mean(dim)
    y_anom = y - y.mean(dim)
    return (y_anom * x_anom).mean(dim) / x_anom.var(dim)


def vertical_integrate(term):
    # Integrate a level-dependent term from 1000 to 700 hPa in Pa.
    term = term.sortby('level').chunk({'level': -1})
    term = term.assign_coords(level=('level', term['level'].values * 100.0))
    return term.integrate('level') / G


def mfc_from_flux(q_u, q_v):
    # Compute MFC on the sphere from vertically integrated flux components.
    q_u, q_v = xr.align(q_u, q_v, join='inner')
    q_u = q_u.sortby('lat')
    q_v = q_v.sortby('lat')

    lat_rad = np.deg2rad(q_u['lat'])
    dqu_dlon = q_u.differentiate('lon') * (180.0 / np.pi)
    dqv_dlat = q_v.differentiate('lat') * (180.0 / np.pi)

    # Positive MFC means convergence.
    return (-(dqu_dlon / (R_EARTH * np.cos(lat_rad)) + dqv_dlat / R_EARTH)).rename('mfc')


## Load ERA5 and Nino3.4

The datasets are opened lazily, then immediately subset to the target box and the 1000-700 hPa layer.


In [ ]:
# Open q, u, and v lazily, then subset early to the target box and lower troposphere.
# remove dtype in attrs on variable 'step'. 
q_ds = xr.open_dataset(Q_PATH, drop_variables=['step'], chunks={'time': 12})[['q']]
rename_map = {}
if 'latitude' in q_ds.coords or 'latitude' in q_ds.dims:
    rename_map['latitude'] = 'lat'
if 'longitude' in q_ds.coords or 'longitude' in q_ds.dims:
    rename_map['longitude'] = 'lon'
if rename_map:
    q_ds = q_ds.rename(rename_map)
q_ds = q_ds.assign_coords(lon=(q_ds.lon % 360))

uv_ds = xr.open_dataset(UV_PATH, drop_variables=['step'], chunks={'time': 12})[['u', 'v']]
rename_map = {}
if 'latitude' in uv_ds.coords or 'latitude' in uv_ds.dims:
    rename_map['latitude'] = 'lat'
if 'longitude' in uv_ds.coords or 'longitude' in uv_ds.dims:
    rename_map['longitude'] = 'lon'
if rename_map:
    uv_ds = uv_ds.rename(rename_map)
uv_ds = uv_ds.assign_coords(lon=(uv_ds.lon % 360))

lat_values = np.asarray(q_ds['lat'].values)
lat_slice = (
    slice(TARGET_BOX['lat_min'], TARGET_BOX['lat_max'])
    if lat_values[0] < lat_values[-1]
    else slice(TARGET_BOX['lat_max'], TARGET_BOX['lat_min'])
)
selected_levels = [float(lev) for lev in np.asarray(q_ds['level'].values) if 700 <= lev <= 1000]

q_ds = q_ds.sel(
    time=slice(START, END),
    level=selected_levels,
    lat=lat_slice,
    lon=slice(TARGET_BOX['lon_min'], TARGET_BOX['lon_max']),
).sortby('lat')

uv_ds = uv_ds.sel(
    time=slice(START, END),
    level=selected_levels,
    lat=lat_slice,
    lon=slice(TARGET_BOX['lon_min'], TARGET_BOX['lon_max']),
).sortby('lat')

q = q_ds['q']
u = uv_ds['u']
v = uv_ds['v']

print('Selected levels (hPa):', selected_levels)
print('q dims:', q.dims, 'shape:', q.shape)
print('u dims:', u.dims, 'shape:', u.shape)
print('v dims:', v.dims, 'shape:', v.shape)


## DJF seasonal means and Nino3.4 standardization

Nino3.4 is standardized using the full DJF analysis period so the P1 and P2 regression coefficients share the same scale.


In [ ]:
q_djf = djf_seasonal_mean(q)
u_djf = djf_seasonal_mean(u)
v_djf = djf_seasonal_mean(v)

nino_df = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', NINO34_COLUMN],
    parse_dates=['Date'],
)
nino_df = nino_df.set_index('Date').loc[START:END].copy()
nino_df[NINO34_COLUMN] = pd.to_numeric(nino_df[NINO34_COLUMN], errors='coerce')
nino_df[NINO34_COLUMN] = nino_df[NINO34_COLUMN].replace(-9999.0, np.nan)
nino_df = nino_df[nino_df.index.month.isin(DJF_MONTHS)].copy()
nino_df['djf_year'] = nino_df.index.year + (nino_df.index.month == 12).astype('int8')

nino_djf = nino_df.groupby('djf_year')[NINO34_COLUMN].mean().loc[FULL_YEARS]
nino_djf = xr.DataArray(
    nino_djf.to_numpy(),
    coords={'djf_year': nino_djf.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino_clim = nino_djf.sel(djf_year=slice(1991, 2020))
nino_djf_std = (nino_djf - nino_clim.mean('djf_year')) / nino_clim.std('djf_year', ddof=0)

print('DJF years in q/u/v:', int(q_djf.sizes['djf_year']))
print('DJF years in Nino3.4:', int(nino_djf_std.sizes['djf_year']))
print('Nino3.4 climatology years:', int(nino_clim.sizes['djf_year']))
print('Standardized Nino3.4 mean (full series):', float(nino_djf_std.mean('djf_year')))
print('Standardized Nino3.4 std (1991-2020 baseline):', float(nino_clim.std('djf_year', ddof=0)))
print('q DJF year range:', int(q_djf.djf_year.values[0]), 'to', int(q_djf.djf_year.values[-1]))


## Period-wise decomposition

For each period, compute the period climatology first, then regress the flux terms onto the standardized Nino3.4 index.


In [ ]:
period_years = {
    'P1': P1_YEARS,
    'P2': P2_YEARS,
}

period_results = {}

for period_name, years in period_years.items():
    q_p = q_djf.sel(djf_year=years)
    u_p = u_djf.sel(djf_year=years)
    v_p = v_djf.sel(djf_year=years)
    n_p = nino_djf_std.sel(djf_year=years)

    q_p, u_p, v_p, n_p = xr.align(q_p, u_p, v_p, n_p, join='inner')

    qbar = q_p.mean('djf_year')
    ubar = u_p.mean('djf_year')
    vbar = v_p.mean('djf_year')

    q_anom = q_p - qbar
    u_anom = u_p - ubar
    v_anom = v_p - vbar

    # Total response: regression of the full moisture flux q*u or q*v.
    total_u = regress_1d(q_p * u_p, n_p)
    total_v = regress_1d(q_p * v_p, n_p)

    # Dynamic response: circulation change acting on the mean moisture field.
    dynamic_u = qbar * regress_1d(u_anom, n_p)
    dynamic_v = qbar * regress_1d(v_anom, n_p)

    # Thermodynamic response: moisture change acting on the mean circulation.
    thermodynamic_u = ubar * regress_1d(q_anom, n_p)
    thermodynamic_v = vbar * regress_1d(q_anom, n_p)

    # Nonlinear response: interaction between moisture and wind anomalies.
    nonlinear_u = regress_1d(q_anom * u_anom, n_p)
    nonlinear_v = regress_1d(q_anom * v_anom, n_p)

    flux_components = {
        'total': (total_u, total_v),
        'dynamic': (dynamic_u, dynamic_v),
        'thermodynamic': (thermodynamic_u, thermodynamic_v),
        'nonlinear': (nonlinear_u, nonlinear_v),
    }

    component_mfc = {}
    component_area_mean = {}

    for component_name, (flux_u, flux_v) in flux_components.items():
        q_u = vertical_integrate(flux_u)
        q_v = vertical_integrate(flux_v)
        mfc = mfc_from_flux(q_u, q_v)
        component_mfc[component_name] = mfc

        weights = np.cos(np.deg2rad(mfc['lat']))
        component_area_mean[component_name] = float(
            mfc.weighted(weights).mean(('lat', 'lon')).compute()
        )

    residual_mfc = (
        component_mfc['total']
        - component_mfc['dynamic']
        - component_mfc['thermodynamic']
        - component_mfc['nonlinear']
    )
    residual_weights = np.cos(np.deg2rad(residual_mfc['lat']))
    component_area_mean['residual'] = float(
        residual_mfc.weighted(residual_weights).mean(('lat', 'lon')).compute()
    )

    period_results[period_name] = component_area_mean
    #fix error AttributeError: 'DataArray' object has no attribute 'abs'
    residual_abs = float(
        np.abs(residual_mfc).weighted(residual_weights).mean(('lat', 'lon')).compute()
    )
    print(
        f'{period_name} closure residual area-mean: '
        f"{component_area_mean['residual']:+.4e} kg m^-2 s^-1 "
        f"({component_area_mean['residual'] * 86400:+.4f} mm/day)"
    )
    print(f'{period_name} weighted abs residual mean: {residual_abs:.4e} kg m^-2 s^-1')


## Export table

The residual row is the closure check: `total - dynamic - thermodynamic - nonlinear`. Values stay in the native `kg m^-2 s^-1` unit.


In [ ]:
term_order = ['total', 'dynamic', 'thermodynamic', 'nonlinear', 'residual']

df_out = pd.DataFrame(
    {
        'P1_kg_m2_s': [period_results['P1'][term] for term in term_order],
        'P2_kg_m2_s': [period_results['P2'][term] for term in term_order],
    },
    index=term_order,
)
df_out['diff_P2_minus_P1_kg_m2_s'] = df_out['P2_kg_m2_s'] - df_out['P1_kg_m2_s']

df_out = df_out[[
    'P1_kg_m2_s',
    'P2_kg_m2_s',
    'diff_P2_minus_P1_kg_m2_s',
]]

df_out.to_csv(OUT_CSV, index_label='term')
print(f'Saved CSV -> {OUT_CSV}')
print(df_out.round(6).to_string())


## Short interpretation

The code below prints the dominant term in each period, the term that changes most between periods, and whether the dynamic term supports a circulation-change interpretation. All values are in `kg m^-2 s^-1`.


In [ ]:
analysis_terms = ['total', 'dynamic', 'thermodynamic', 'nonlinear']

p1_dom = df_out.loc[analysis_terms, 'P1_kg_m2_s'].abs().idxmax()
p2_dom = df_out.loc[analysis_terms, 'P2_kg_m2_s'].abs().idxmax()
change_dom = df_out.loc[analysis_terms, 'diff_P2_minus_P1_kg_m2_s'].abs().idxmax()

total_change = float(df_out.loc['total', 'diff_P2_minus_P1_kg_m2_s'])
dynamic_change = float(df_out.loc['dynamic', 'diff_P2_minus_P1_kg_m2_s'])
thermodynamic_change = float(df_out.loc['thermodynamic', 'diff_P2_minus_P1_kg_m2_s'])
nonlinear_change = float(df_out.loc['nonlinear', 'diff_P2_minus_P1_kg_m2_s'])

if np.isclose(total_change, 0.0):
    support_text = (
        'Total change is near zero, so the circulation-change interpretation is not decisive.'
    )
else:
    dynamic_supports = (
        np.sign(dynamic_change) == np.sign(total_change)
    ) and (
        abs(dynamic_change) >= max(abs(thermodynamic_change), abs(nonlinear_change))
    )
    if dynamic_supports:
        support_text = (
            'Dynamic term supports the circulation-change hypothesis: it moves in the same '
            'direction as the total change and is the largest shift among the physical terms.'
        )
    else:
        support_text = (
            'Dynamic term does not clearly dominate the change; thermodynamic and/or '
            'nonlinear terms contribute comparably or more.'
        )

print(f'P1 dominant term: {p1_dom}')
print(f'P2 dominant term: {p2_dom}')
print(f'Largest change from P1 to P2: {change_dom}')
print(support_text)
print(f'Total change: {total_change:+.6e} kg m^-2 s^-1')
print(f'Dynamic change: {dynamic_change:+.6e} kg m^-2 s^-1')
print(f'Thermodynamic change: {thermodynamic_change:+.6e} kg m^-2 s^-1')
print(f'Nonlinear change: {nonlinear_change:+.6e} kg m^-2 s^-1')


## Bar plot


In [ ]:
plot_terms = ['total', 'dynamic', 'thermodynamic', 'nonlinear']
plot_labels = ['Total', 'Dynamic', 'Thermodynamic', 'Nonlinear']

x = np.arange(len(plot_terms))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
series = [
    ('P1', 'P1_kg_m2_s', -width, '#4C78A8'),
    ('P2', 'P2_kg_m2_s', 0.0, '#F58518'),
    ('P2-P1', 'diff_P2_minus_P1_kg_m2_s', width, '#54A24B'),
]
for label, col, offset, color in series:
    ax.bar(
        x + offset,
        df_out.loc[plot_terms, col].to_numpy(),
        width=width,
        label=label,
        color=color,
    )

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot_labels)
ax.set_ylabel('kg m$^{-2}$ s$^{-1}$ / 1\u03C3 Nino3.4')
ax.set_title('Area-mean ENSO-related MFC decomposition over TARGET_BOX')
ax.legend(frameon=False, ncol=3)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(OUT_PNG, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved figure -> {OUT_PNG}')
